In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
DATA_DIR = '/content/drive/MyDrive/dataset/mrnet_all'
BATCH_SIZE = 1
NUM_EPOCHS = 35
RANDOM_SEED = 42
MAX_SLICES = 40          # Single view at a time, same as V11
LR = 1e-4
WEIGHT_DECAY = 0.01
PATIENCE = 10
DROPOUT = 0.3

# Loss weights for each task
TASK_WEIGHT_ACL = 1.0
TASK_WEIGHT_MENISCUS = 1.0
TASK_WEIGHT_ABNORMAL = 0.5

# Model selection weights (for composite AUC)
SEL_WEIGHT_ACL = 0.5
SEL_WEIGHT_MEN = 0.3
SEL_WEIGHT_ABN = 0.2

# Which views to train (set to subset for testing)
VIEWS = ['sagittal', 'coronal', 'axial']


In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve, classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'  GPU: {torch.cuda.get_device_name(0)}')


Device: cuda
  GPU: Tesla T4


In [ ]:
data_path = Path(DATA_DIR)
metadata = pd.read_csv(data_path / 'metadata.csv')

print(f"Total patients: {len(metadata)}")
print(f"\nLabel distribution:")
print(f"  ACL tear:      {metadata.label_acl.sum():4d} / {len(metadata)} ({100*metadata.label_acl.mean():.1f}%)")
print(f"  Meniscus tear: {metadata.label_meniscus.sum():4d} / {len(metadata)} ({100*metadata.label_meniscus.mean():.1f}%)")
print(f"  Abnormal:      {metadata.label_abnormal.sum():4d} / {len(metadata)} ({100*metadata.label_abnormal.mean():.1f}%)")

# Stratified split on ACL label (primary task)
train_df, val_df = train_test_split(
    metadata, test_size=0.15, random_state=RANDOM_SEED,
    stratify=metadata['label_acl']
)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print(f"\nTrain: {len(train_df)} patients")
print(f"  ACL: {train_df.label_acl.sum()} tear / {(train_df.label_acl==0).sum()} normal")
print(f"Val: {len(val_df)} patients")
print(f"  ACL: {val_df.label_acl.sum()} tear / {(val_df.label_acl==0).sum()} normal")


Total patients: 1250

Label distribution:
  ACL tear:       262 / 1250 (21.0%)
  Meniscus tear:  449 / 1250 (35.9%)
  Abnormal:      1008 / 1250 (80.6%)

Train: 1062 patients
  ACL: 223 tear / 839 normal
Val: 188 patients
  ACL: 39 tear / 149 normal


In [ ]:
class SingleViewDataset(Dataset):
    """Loads ONE view at a time from mrnet_all .npz files with 3 labels.
    This is the MRNet-faithful approach: one model per view.
    """
    def __init__(self, df, data_dir, view='sagittal', max_slices=MAX_SLICES, augment=False):
        self.df = df.reset_index(drop=True)
        self.data_dir = Path(data_dir)
        self.view = view
        self.max_slices = max_slices
        self.augment = augment
        self.valid_indices = []
        for idx in range(len(self.df)):
            fpath = self.data_dir / self.df.iloc[idx]['filename']
            if fpath.exists():
                self.valid_indices.append(idx)
        print(f"  [{view}] {len(self.valid_indices)} valid patients (of {len(self.df)})")

    def __len__(self):
        return len(self.valid_indices)

    def __getitem__(self, idx):
        patient_idx = self.valid_indices[idx]
        row = self.df.iloc[patient_idx]
        data = np.load(self.data_dir / row['filename'])
        volume = data[self.view]  # (S, 256, 256), uint8
        slices = volume.astype(np.float32) / 255.0

        # Center crop if too many slices
        if slices.shape[0] > self.max_slices:
            offset = (slices.shape[0] - self.max_slices) // 2
            slices = slices[offset:offset + self.max_slices]

        # Simple augmentation: random horizontal flip
        if self.augment and np.random.random() > 0.5:
            slices = slices[:, :, ::-1].copy()

        # Grayscale -> 3-channel: (S, H, W) -> (S, 3, H, W)
        slices_3ch = np.stack((slices,)*3, axis=1)
        tensor = torch.FloatTensor(slices_3ch)

        labels = torch.LongTensor([
            int(row['label_acl']),
            int(row['label_meniscus']),
            int(row['label_abnormal'])
        ])
        return tensor, labels

    def get_label_counts(self):
        acl = [int(self.df.iloc[i]['label_acl']) for i in self.valid_indices]
        men = [int(self.df.iloc[i]['label_meniscus']) for i in self.valid_indices]
        abn = [int(self.df.iloc[i]['label_abnormal']) for i in self.valid_indices]
        return {'acl': acl, 'meniscus': men, 'abnormal': abn}


In [ ]:
_tmp = SingleViewDataset(train_df, DATA_DIR, view='sagittal')
label_counts = _tmp.get_label_counts()

def compute_weight(labels):
    n = len(labels)
    n_pos = sum(labels)
    n_neg = n - n_pos
    w_neg = n / (2 * n_neg) if n_neg > 0 else 1.0
    w_pos = n / (2 * n_pos) if n_pos > 0 else 1.0
    return torch.FloatTensor([w_neg, w_pos]).to(device)

weight_acl = compute_weight(label_counts['acl'])
weight_meniscus = compute_weight(label_counts['meniscus'])
weight_abnormal = compute_weight(label_counts['abnormal'])

print(f"Class weights:")
print(f"  ACL:      Normal={weight_acl[0]:.3f}, Tear={weight_acl[1]:.3f}")
print(f"  Meniscus: Normal={weight_meniscus[0]:.3f}, Tear={weight_meniscus[1]:.3f}")
print(f"  Abnormal: Normal={weight_abnormal[0]:.3f}, Abnormal={weight_abnormal[1]:.3f}")


  [sagittal] 1062 valid patients (of 1062)
Class weights:
  ACL:      Normal=0.633, Tear=2.381
  Meniscus: Normal=0.774, Tear=1.412
  Abnormal: Normal=2.541, Abnormal=0.623


In [ ]:
BLOCK_SIZE = 5  # add to config cell

class MRNetBlockAttn(nn.Module):
    """Multi-task MRNet with Block Attention pooling."""
    def __init__(self, dropout=DROPOUT, block_size=BLOCK_SIZE, attn_hidden=128):
        super().__init__()
        backbone = models.efficientnet_b0(weights='IMAGENET1K_V1')
        self.features = backbone.features
        self.pool = backbone.avgpool
        self.drop = nn.Dropout(p=dropout)
        self.block_size = block_size

        # Attention over blocks (not individual slices — much fewer tokens)
        self.block_attn = nn.Sequential(
            nn.Linear(1280, attn_hidden),
            nn.Tanh(),
            nn.Dropout(0.25),
            nn.Linear(attn_hidden, 1)
        )

        self.head_acl = nn.Linear(1280, 2)
        self.head_meniscus = nn.Linear(1280, 2)
        self.head_abnormal = nn.Linear(1280, 2)

        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        total = sum(p.numel() for p in self.parameters())
        print(f"  Backbone: EfficientNet-B0 (all trainable)")
        print(f"  Pooling: Block Attention (block_size={block_size}, hidden={attn_hidden})")
        print(f"  Heads: ACL(1280->2), Men(1280->2), Abn(1280->2)")
        print(f"  Params: {trainable:,} trainable / {total:,} total")

    def forward(self, x):
        x = x.squeeze(0)                        # (S, 3, H, W)
        features = self.features(x)              # (S, 1280, 8, 8)
        pooled = self.pool(features).flatten(1)  # (S, 1280)

        S = pooled.shape[0]
        # Pad so S is divisible by block_size
        remainder = S % self.block_size
        if remainder != 0:
            pad_size = self.block_size - remainder
            pooled = torch.cat([pooled, pooled[-pad_size:]], dim=0)  # repeat last slices

        # Reshape into blocks and max-pool within each block
        n_blocks = pooled.shape[0] // self.block_size
        blocks = pooled.view(n_blocks, self.block_size, -1)  # (B, block_size, 1280)
        block_feats = blocks.max(dim=1)[0]                    # (B, 1280)

        # Attention across blocks
        attn_scores = self.block_attn(block_feats)             # (B, 1)
        attn_weights = torch.softmax(attn_scores, dim=0)       # (B, 1)
        volume_feat = (attn_weights * block_feats).sum(dim=0, keepdim=True)  # (1, 1280)

        volume_feat = self.drop(volume_feat)
        return (self.head_acl(volume_feat),
                self.head_meniscus(volume_feat),
                self.head_abnormal(volume_feat))


In [ ]:
def train_epoch(model, loader, optimizer, crits, device):
    model.train()
    total_loss = 0
    all_labels = {'acl': [], 'meniscus': [], 'abnormal': []}
    all_probs = {'acl': [], 'meniscus': [], 'abnormal': []}

    for volumes, labels in tqdm(loader, desc='Training', leave=False):
        volumes = volumes.to(device)
        lab_acl = labels[:, 0].to(device)
        lab_men = labels[:, 1].to(device)
        lab_abn = labels[:, 2].to(device)

        optimizer.zero_grad()
        out_acl, out_men, out_abn = model(volumes.float())

        loss = (TASK_WEIGHT_ACL * crits[0](out_acl, lab_acl) +
                TASK_WEIGHT_MENISCUS * crits[1](out_men, lab_men) +
                TASK_WEIGHT_ABNORMAL * crits[2](out_abn, lab_abn))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

        all_labels['acl'].extend(lab_acl.cpu().numpy())
        all_labels['meniscus'].extend(lab_men.cpu().numpy())
        all_labels['abnormal'].extend(lab_abn.cpu().numpy())
        all_probs['acl'].extend(torch.softmax(out_acl, 1)[:, 1].detach().cpu().numpy())
        all_probs['meniscus'].extend(torch.softmax(out_men, 1)[:, 1].detach().cpu().numpy())
        all_probs['abnormal'].extend(torch.softmax(out_abn, 1)[:, 1].detach().cpu().numpy())

    avg_loss = total_loss / len(loader)
    aucs = {}
    for task in ['acl', 'meniscus', 'abnormal']:
        try:
            aucs[task] = roc_auc_score(all_labels[task], all_probs[task])
        except ValueError:
            aucs[task] = 0.5
    return avg_loss, aucs


def validate(model, loader, crits, device):
    model.eval()
    total_loss = 0
    all_labels = {'acl': [], 'meniscus': [], 'abnormal': []}
    all_probs = {'acl': [], 'meniscus': [], 'abnormal': []}

    with torch.no_grad():
        for volumes, labels in tqdm(loader, desc='Validating', leave=False):
            volumes = volumes.to(device)
            lab_acl = labels[:, 0].to(device)
            lab_men = labels[:, 1].to(device)
            lab_abn = labels[:, 2].to(device)

            out_acl, out_men, out_abn = model(volumes.float())
            loss = (TASK_WEIGHT_ACL * crits[0](out_acl, lab_acl) +
                    TASK_WEIGHT_MENISCUS * crits[1](out_men, lab_men) +
                    TASK_WEIGHT_ABNORMAL * crits[2](out_abn, lab_abn))
            total_loss += loss.item()

            all_labels['acl'].extend(lab_acl.cpu().numpy())
            all_labels['meniscus'].extend(lab_men.cpu().numpy())
            all_labels['abnormal'].extend(lab_abn.cpu().numpy())
            all_probs['acl'].extend(torch.softmax(out_acl, 1)[:, 1].cpu().numpy())
            all_probs['meniscus'].extend(torch.softmax(out_men, 1)[:, 1].cpu().numpy())
            all_probs['abnormal'].extend(torch.softmax(out_abn, 1)[:, 1].cpu().numpy())

    avg_loss = total_loss / len(loader)
    aucs = {}
    for task in ['acl', 'meniscus', 'abnormal']:
        try:
            aucs[task] = roc_auc_score(all_labels[task], all_probs[task])
        except ValueError:
            aucs[task] = 0.5
    return avg_loss, aucs, all_labels, all_probs


In [ ]:
view_results = {}  # Store per-view val predictions

for view in VIEWS:
    print(f"\n{'='*60}")
    print(f"  TRAINING VIEW: {view.upper()}")
    print(f"{'='*60}")

    # Create datasets for this view
    print("Creating datasets...")
    train_dataset = SingleViewDataset(train_df, DATA_DIR, view=view, augment=True)
    val_dataset = SingleViewDataset(val_df, DATA_DIR, view=view, augment=False)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=2, pin_memory=True)

    # Create model, optimizer, scheduler
    print("Creating model...")
    model = MRNetPerView(dropout=DROPOUT).to(device)
    crits = [
        nn.CrossEntropyLoss(weight=weight_acl),
        nn.CrossEntropyLoss(weight=weight_meniscus),
        nn.CrossEntropyLoss(weight=weight_abnormal),
    ]
    optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.3, patience=5, min_lr=1e-7)

    # Training loop
    best_composite = 0.0
    no_improve = 0
    save_path = f'/content/drive/MyDrive/dataset/best_v16_{view}.pth'

    for epoch in range(NUM_EPOCHS):
        lr = optimizer.param_groups[0]['lr']
        train_loss, train_aucs = train_epoch(model, train_loader, optimizer, crits, device)
        val_loss, val_aucs, val_labels, val_probs = validate(model, val_loader, crits, device)

        composite_val = (SEL_WEIGHT_ACL * val_aucs['acl'] +
                         SEL_WEIGHT_MEN * val_aucs['meniscus'] +
                         SEL_WEIGHT_ABN * val_aucs['abnormal'])
        scheduler.step(composite_val)

        gap = 100 * ((SEL_WEIGHT_ACL * train_aucs['acl'] + SEL_WEIGHT_MEN * train_aucs['meniscus'] +
                       SEL_WEIGHT_ABN * train_aucs['abnormal']) - composite_val)

        print(f"[{view}] Epoch {epoch+1}/{NUM_EPOCHS}  (lr={lr:.1e})")
        print(f"  Train: loss={train_loss:.4f}  ACL={train_aucs['acl']:.4f}  Men={train_aucs['meniscus']:.4f}  Abn={train_aucs['abnormal']:.4f}")
        print(f"  Val:   loss={val_loss:.4f}  ACL={val_aucs['acl']:.4f}  Men={val_aucs['meniscus']:.4f}  Abn={val_aucs['abnormal']:.4f}  (gap={gap:.1f}%)")

        if composite_val > best_composite:
            best_composite = composite_val
            no_improve = 0
            torch.save(model.state_dict(), save_path)
            print(f"  -> Saved best {view} (Comp={best_composite:.4f})")
        else:
            no_improve += 1
            print(f"  -> No improvement ({no_improve}/{PATIENCE})")
            if no_improve >= PATIENCE:
                print(f"  Early stopping {view} at epoch {epoch+1}")
                break

    # Load best and collect val predictions
    model.load_state_dict(torch.load(save_path, map_location=device, weights_only=True))
    _, _, val_labels, val_probs = validate(model, val_loader, crits, device)
    view_results[view] = {'labels': val_labels, 'probs': val_probs, 'aucs': {
        t: roc_auc_score(val_labels[t], val_probs[t]) for t in ['acl', 'meniscus', 'abnormal']
    }}
    print(f"\n  Best {view}: ACL={view_results[view]['aucs']['acl']:.4f}, "
          f"Men={view_results[view]['aucs']['meniscus']:.4f}, "
          f"Abn={view_results[view]['aucs']['abnormal']:.4f}")

    del model, optimizer, scheduler
    torch.cuda.empty_cache()




  TRAINING VIEW: SAGITTAL
Creating datasets...
  [sagittal] 1062 valid patients (of 1062)
  [sagittal] 188 valid patients (of 188)
Creating model...
Downloading: "https://download.pytorch.org/models/efficientnet_b1_rwightman-bac287d4.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b1_rwightman-bac287d4.pth


100%|██████████| 30.1M/30.1M [00:00<00:00, 192MB/s]


  Backbone: EfficientNet-B1 (all trainable)
  Heads: ACL(1280->2), Men(1280->2), Abn(1280->2)
  Params: 6,520,870 trainable / 6,520,870 total


Training:   0%|          | 0/1062 [00:00<?, ?it/s]

Validating:   0%|          | 0/188 [00:00<?, ?it/s]

...

Training:   0%|          | 0/1062 [00:00<?, ?it/s]

Validating:   0%|          | 0/188 [00:00<?, ?it/s]

[axial] Epoch 10/35  (lr=1.0e-04)
  Train: loss=0.6003  ACL=0.9832  Men=0.9268  Abn=0.9266
  Val:   loss=2.4857  ACL=0.8227  Men=0.6530  Abn=0.7365  (gap=20.0%)
  -> No improvement (1/10)


Training:   0%|          | 0/1062 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# %% [markdown]
# # Cell 9B: Resume — Load Pre-trained View Models

# %%
print("Loading pre-trained view models from Drive...\n")
view_results = {}

for view in VIEWS:
    save_path = f'/content/drive/MyDrive/dataset/best_v18_block_{view}.pth'
    print(f"--- {view.upper()} ---")
    print(f"  Loading: {save_path}")

    # Create dataset + loader for this view
    val_dataset = SingleViewDataset(val_df, DATA_DIR, view=view, augment=False)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=2, pin_memory=True)

    # Load model
    model = MRNetPerView(dropout=DROPOUT).to(device)
    model.load_state_dict(torch.load(save_path, map_location=device, weights_only=True))

    # Run validation to collect predictions
    crits = [
        nn.CrossEntropyLoss(weight=weight_acl),
        nn.CrossEntropyLoss(weight=weight_meniscus),
        nn.CrossEntropyLoss(weight=weight_abnormal),
    ]
    _, _, val_labels, val_probs = validate(model, val_loader, crits, device)

    view_results[view] = {
        'labels': val_labels,
        'probs': val_probs,
        'aucs': {t: roc_auc_score(val_labels[t], val_probs[t])
                 for t in ['acl', 'meniscus', 'abnormal']}
    }
    a = view_results[view]['aucs']
    print(f"  ACL={a['acl']:.4f}  Men={a['meniscus']:.4f}  Abn={a['abnormal']:.4f}\n")

    del model
    torch.cuda.empty_cache()

print("✓ All views loaded. Run Cell 10 and 11 next.")


Loading pre-trained view models from Drive...

--- SAGITTAL ---
  Loading: /content/drive/MyDrive/dataset/best_v18_sagittal.pth
  [sagittal] 188 valid patients (of 188)
  Backbone: EfficientNet-B1 (all trainable)
  Heads: ACL(1280->2), Men(1280->2), Abn(1280->2)
  Params: 6,520,870 trainable / 6,520,870 total


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/dataset/best_v18_sagittal.pth'

In [ ]:
# Cell 9B: Resume — Load saved view models and rebuild view_results
print("Loading pre-trained view models from Drive...\n")
view_results = {}
ALL_VIEWS = ['sagittal', 'coronal', 'axial']

crits = [
    nn.CrossEntropyLoss(weight=weight_acl),
    nn.CrossEntropyLoss(weight=weight_meniscus),
    nn.CrossEntropyLoss(weight=weight_abnormal),
]

for view in ALL_VIEWS:
    save_path = f'/content/drive/MyDrive/dataset/best_v18_{view}.pth'
    print(f"--- {view.upper()} ---")

    val_dataset = SingleViewDataset(val_df, DATA_DIR, view=view, augment=False)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=2, pin_memory=True)

    model = MRNetPerView(dropout=DROPOUT).to(device)
    model.load_state_dict(torch.load(save_path, map_location=device, weights_only=True))

    _, _, val_labels, val_probs = validate(model, val_loader, crits, device)

    view_results[view] = {
        'labels': val_labels,
        'probs': val_probs,
        'aucs': {t: roc_auc_score(val_labels[t], val_probs[t])
                 for t in ['acl', 'meniscus', 'abnormal']}
    }
    a = view_results[view]['aucs']
    print(f"  ACL={a['acl']:.4f}  Men={a['meniscus']:.4f}  Abn={a['abnormal']:.4f}\n")
    del model
    torch.cuda.empty_cache()

print("✓ Ready — run Cell 10 next.")


Loading pre-trained view models from Drive...

--- SAGITTAL ---
  [sagittal] 188 valid patients (of 188)
  Backbone: EfficientNet-B1 (all trainable)
  Heads: ACL(1280->2), Men(1280->2), Abn(1280->2)
  Params: 6,520,870 trainable / 6,520,870 total


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/dataset/best_v18_sagittal.pth'

In [ ]:
ALL_VIEWS = ['sagittal', 'coronal', 'axial']

print("\n" + "="*60)
print("  COMBINING VIEWS (MRNet-style)")
print("="*60)

# Per-view AUC summary
for view in ALL_VIEWS:
    a = view_results[view]['aucs']
    print(f"  {view:10s}: ACL={a['acl']:.4f}  Men={a['meniscus']:.4f}  Abn={a['abnormal']:.4f}")

# Simple average combination (no extra training needed)
print("\n--- Method 1: Simple Average ---")
combined_probs = {}
for task in ['acl', 'meniscus', 'abnormal']:
    combined_probs[task] = np.mean(
        [view_results[v]['probs'][task] for v in ALL_VIEWS], axis=0
    )
    labels = view_results[ALL_VIEWS[0]]['labels'][task]
    auc = roc_auc_score(labels, combined_probs[task])
    print(f"  {task:10s} AUC = {auc:.4f}")

composite = (SEL_WEIGHT_ACL * roc_auc_score(labels, combined_probs['acl']) +
             SEL_WEIGHT_MEN * roc_auc_score(view_results[ALL_VIEWS[0]]['labels']['meniscus'], combined_probs['meniscus']) +
             SEL_WEIGHT_ABN * roc_auc_score(view_results[ALL_VIEWS[0]]['labels']['abnormal'], combined_probs['abnormal']))
print(f"  Composite = {composite:.4f}")

# Learned combination via logistic regression (MRNet paper approach)
print("\n--- Method 2: Logistic Regression (MRNet paper) ---")
for task in ['acl', 'meniscus', 'abnormal']:
    X = np.column_stack([view_results[v]['probs'][task] for v in ALL_VIEWS])
    y = np.array(view_results[ALL_VIEWS[0]]['labels'][task])
    lr_model = LogisticRegression(random_state=RANDOM_SEED)
    lr_model.fit(X, y)
    lr_probs = lr_model.predict_proba(X)[:, 1]
    auc = roc_auc_score(y, lr_probs)
    print(f"  {task:10s} AUC = {auc:.4f}  (weights: {', '.join(f'{v}={w:.3f}' for v, w in zip(ALL_VIEWS, lr_model.coef_[0]))})")



  COMBINING VIEWS (MRNet-style)
  sagittal  : ACL=0.8735  Men=0.7833  Abn=0.7898
  coronal   : ACL=0.9367  Men=0.8655  Abn=0.8739


KeyError: 'axial'

In [ ]:
print('\n' + '=' * 60)
print('RESULTS — V16 MRNet (Per-View Multi-Task)')
print('=' * 60)

# Use simple average as final predictions
label_names = ['Normal', 'Tear']
for task, title in [('acl', 'ACL'), ('meniscus', 'Meniscus'), ('abnormal', 'Abnormal')]:
    labels = view_results[ALL_VIEWS[0]]['labels'][task]
    probs = combined_probs[task]
    auc = roc_auc_score(labels, probs)
    fpr, tpr, thresholds = roc_curve(labels, probs)
    j = tpr - fpr
    best_idx = np.argmax(j)
    print(f"\n{title}: AUC={auc:.4f}, Threshold={thresholds[best_idx]:.4f}, "
          f"Sens={tpr[best_idx]:.3f}, Spec={1-fpr[best_idx]:.3f}")
    preds = [1 if p >= 0.5 else 0 for p in probs]
    print(classification_report(labels, preds, target_names=label_names, digits=3))

# Confusion matrices
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, task, title in zip(axes, ['acl', 'meniscus', 'abnormal'],
                            ['ACL Tear', 'Meniscus Tear', 'Abnormality']):
    labels = view_results[ALL_VIEWS[0]]['labels'][task]
    probs = combined_probs[task]
    preds = [1 if p >= 0.5 else 0 for p in probs]
    cm = confusion_matrix(labels, preds)
    auc = roc_auc_score(labels, probs)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Normal', 'Positive'], yticklabels=['Normal', 'Positive'], ax=ax)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.set_title(f'{title} (AUC={auc:.3f})')

plt.suptitle('V16 MRNet Per-View Multi-Task — Confusion Matrices', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/dataset/confusion_matrix_v16.png', dpi=150)
plt.show()


RESULTS — V16 MRNet (Per-View Multi-Task)


NameError: name 'combined_probs' is not defined